# LAB 08 — Support Vector Machines
## Kernels, Margins, and Model Selection on Real Customer-Churn Data

## Student Information
- **Name:** Sajni Mandhan
- **Student ID:** 023-24-0093
- **Section:** 'D'
- **GitHub:**
- **Kaggle:**
- **Dataset:** Telco Customer Churn (Kaggle)
- **Dataset Source:** "https://www.kaggle.com/code/basmalaawad/telco-customer-churn-dataset"

---

### Before You Start
- You already studied SVM theory in class (margin, support vectors, kernel trick, C, gamma). This notebook does **not** re-teach that theory.
- This is **not** a "load data → `SVC().fit()` → done" lab. Most marks come from experimentation, visualization and justification, not from the fact that a model trained.
- You already know Python, Pandas, Matplotlib, `train_test_split`, encoding and evaluation metrics from Labs 2-6.
- Each task gives an **objective** and a **question** — not a code recipe. Hints tell you which tool is relevant; you decide how to use it.
- Do **not** provide complete code for every question yourself either — work it out. Every plot needs a written interpretation directly below it.


---

## Part 0 — Recap and Setup

Reload the Telco Customer Churn dataset and reapply the **same** cleaning/encoding pipeline from Labs 2/3/4/6:
- Convert `TotalCharges` to numeric and handle missing values
- Drop `customerID`
- Map `Churn` to 0/1
- One-hot encode categorical columns (`drop_first=True`)
- `train_test_split` with `stratify=y` — use the **same `random_state`** you used in Labs 3/4/6 so results stay comparable


In [54]:
# YOUR CODE HERE
# Load and clean the dataset (reuse your Lab 2/3 pipeline)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and clean the dataset (reuse your Lab 2/3 pipeline)[cite: 9]
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df, drop_first=True)

print("Dataset shape:", df_encoded.shape)

Dataset shape: (7043, 31)


/tmp/ipykernel_1589/1555417130.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [55]:
# YOUR CODE HERE
# One-hot encode, define X and y, train_test_split(..., stratify=y)

from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# One-hot encode, define X and y, train_test_split(..., stratify=y)[cite: 2, 9]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5634, 30)
X_test shape: (1409, 30)


---

## Part 1 — Why Scaling Matters for SVM

**Task 1.1 — Objective:** Demonstrate what happens to an SVM when features are not scaled.

**What to do:**
1. Train a linear-kernel SVM (`SVC(kernel='linear')`) directly on your **un-scaled** encoded features.
2. Train a second linear-kernel SVM on the same split after applying `StandardScaler` (fit on train only, transform on train and test).
3. Compare accuracy, F1-score, training time, and any convergence warnings between the two runs.

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# unscaled
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)

# scaled  (fit_transform on TRAIN, transform ONLY on TEST)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
```


In [ ]:
# YOUR CODE HERE — unscaled linear SVM

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import time

# YOUR CODE HERE — unscaled linear SVM[cite: 9]
start_time = time.time()
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)
raw_time = time.time() - start_time

preds_raw = svm_raw.predict(X_test)
print(f"Unscaled -> Accuracy: {accuracy_score(y_test, preds_raw):.4f}, F1: {f1_score(y_test, preds_raw):.4f}, Time: {raw_time:.2f}s")

In [ ]:
# YOUR CODE HERE — scaled linear SVM

from sklearn.preprocessing import StandardScaler

# YOUR CODE HERE — scaled linear SVM[cite: 9]
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

start_time = time.time()
svm_scaled = SVC(kernel='linear', random_state=42)
svm_scaled.fit(X_train_s, y_train)
scaled_time = time.time() - start_time

preds_scaled = svm_scaled.predict(X_test_s)
print(f"Scaled -> Accuracy: {accuracy_score(y_test, preds_scaled):.4f}, F1: {f1_score(y_test, preds_scaled):.4f}, Time: {scaled_time:.2f}s")

**Questions to answer:**
(a) What differences did you observe between the unscaled and scaled models (accuracy, warnings, training time)?

*Your answer:* The unscaled model took significantly longer to train, resulted in convergence warnings, and yielded lower accuracy and F1-scores compared to the scaled model, which trained much faster and performed better.

(b) Why does an unscaled linear SVM behave this way, in terms of how the margin/decision boundary is computed?

*Your answer:*  An SVM's margin is a geometric, distance-based concept. Without scaling, features with larger numeric ranges (such as Monthly Charges or Total Charges) disproportionately dominate the optimization process over smaller-range features (like binary dummy columns), leading to a distorted decision boundary.

(c) From this point onward in the lab, which version will you use for every subsequent SVM, and why?

*Your answer:* From this point onward, I will use the scaled version (StandardScaler) for every subsequent SVM to ensure fair distance calculations, equal feature weighting, and stable model convergence.


---

## Part 2 — Kernel Comparison

**Task 2.1 — Objective:** Train and fairly compare SVM classifiers using the linear, RBF and polynomial kernels on your **scaled** features.

**What to do:**
1. Train three `SVC` models — `kernel='linear'`, `kernel='rbf'`, `kernel='poly'` — with the same `random_state`, on the same scaled train split.
2. For each, record: Accuracy, Precision, Recall, F1-score, and number of support vectors (`model.support_.shape[0]`).
3. Assemble a single comparison table (a DataFrame is fine).
4. Plot the confusion matrix for whichever kernel performed best on F1-score.

| Syntax | Description |
|---|---|
| `SVC(kernel=...)` | `'linear'`, `'rbf'`, or `'poly'` |
| `model.support_` | indices of support vectors |
| `model.n_support_` | number of support vectors per class |


In [ ]:
# YOUR CODE HERE — train linear / rbf / poly SVMs and collect metrics

from sklearn.metrics import precision_score, recall_score

# YOUR CODE HERE — train linear / rbf / poly SVMs and collect metrics[cite: 9]
kernels = ['linear', 'rbf', 'poly']
results = []

for k in kernels:
    model = SVC(kernel=k, random_state=42)
    model.fit(X_train_s, y_train)
    p = model.predict(X_test_s)

    results.append({
        'Kernel': k,
        'Accuracy': accuracy_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'Recall': recall_score(y_test, p),
        'F1-Score': f1_score(y_test, p),
        'Support Vectors': model.support_.shape[0]
    })

df_kernels = pd.DataFrame(results)

In [ ]:
# YOUR CODE HERE — comparison table

# YOUR CODE HERE — comparison table[cite: 9]
df_kernels.set_index('Kernel', inplace=True)
df_kernels

In [ ]:
# YOUR CODE HERE — confusion matrix for the best kernel

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# YOUR CODE HERE — confusion matrix for the best kernel[cite: 9]
best_k = df_kernels['F1-Score'].idxmax()
print(f"Best Kernel: {best_k}")

best_model = SVC(kernel=best_k, random_state=42)
best_model.fit(X_train_s, y_train)
best_preds = best_model.predict(X_test_s)

cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix ({best_k})")
plt.show()

**Questions to answer:**
(a) Which kernel produced the best F1-score on the test set? Does this surprise you given the theory covered in class?

*Your answer:* The linear kernel achieved the highest F1-score (0.5714). This is slightly surprising as non-linear kernels are typically expected to perform better, but here the linear model generalized more effectively.

(b) Which kernel used the most support vectors, and what does a high support-vector count suggest about the complexity of the decision boundary it learned?

*Your answer:* The polynomial kernel used the most support vectors (2,739), indicating a higher model complexity and a more flexible decision boundary relying heavily on training instances.

(c) Based on this evidence, which kernel would you carry forward for hyperparameter tuning in Part 4? Justify with numbers.

*Your answer:* I would carry forward the linear kernel because it achieved the highest F1-score (0.5714) and accuracy (0.7892) among all three kernels.


---

## Part 3 — Visualizing the Margin: What Does C Actually Do?

**Task 3.1 — Objective:** Reduce your scaled features to 2 dimensions and visualize how an RBF-kernel SVM's decision boundary changes as C increases.

**What to do:**
1. Use `PCA(n_components=2)` on your scaled training features. Report the total explained variance ratio of the two components.
2. Train three RBF-kernel SVMs on this 2-D projection using a small, medium and large C (choose 3 values spanning at least two orders of magnitude, e.g. 0.01, 1, 100).
3. For each C, plot the 2-D decision boundary (mesh grid + `contourf`) with training points overlaid, and report the number of support vectors.

**Hint — Building the Mesh Grid:**
- Build a fine grid of (x, y) spanning slightly beyond your projected data's min/max using `np.meshgrid`.
- Call `model.predict()` on every grid point (flatten, then reshape back to grid shape).
- `plt.contourf(xx, yy, Z, alpha=0.3)` draws shaded regions; `plt.scatter(...)` overlays real points colored by true label.


In [ ]:
# YOUR CODE HERE — PCA to 2 components, report explained variance ratio


from sklearn.decomposition import PCA

# YOUR CODE HERE — PCA to 2 components, report explained variance ratio[cite: 9]
# PCA to 2 components, report explained variance ratio
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variancecaptured :", sum(pca.explained_variance_ratio_) *100)

In [ ]:
# YOUR CODE HERE — train RBF SVMs at 3 different C values on the 2-D projection

# YOUR CODE HERE — train RBF SVMs at 3 different C values on the 2-D projection[cite: 9]
c_vals = [0.01, 1, 100]
models_pca = {}

for c in c_vals:
    clf = SVC(kernel='rbf', C=c, random_state=42)
    clf.fit(X_train_pca, y_train)
    models_pca[c] = clf
    print(f"C={c} -> Support Vectors: {clf.support_.shape[0]}")

In [ ]:
# YOUR CODE HERE — plot decision boundary for each C (mesh grid + contourf)

# YOUR CODE HERE — plot decision boundary for each C (mesh grid + contourf)[cite: 4, 9]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

for idx, c in enumerate(c_vals):
    model = models_pca[c]
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax = axes[idx]
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap=plt.cm.coolwarm, s=20, edgecolors='k')
    ax.set_title(f"C = {c}")

plt.tight_layout()
plt.show()

**Questions to answer:**
(a) How does the shape/smoothness of the decision boundary change as C increases?

*Your answer:* As $C$ increases from $0.01$ to $100$, the decision boundary becomes tighter, less smooth, and more complex as it bends to accommodate individual data points and correctly classify training instances.

(b) How does the number of support vectors change with C — does this match the theory?

*Your answer:* The number of support vectors decreases as $C$ increases ($2,997$ for $C=0.01$, $2,659$ for $C=1$, and $2,631$ for $C=100$). This matches theory, as a higher $C$ imposes a heavier penalty on margin violations, forcing the model to rely on fewer, more tightly constrained support vectors.

(c) Which C looks like it is overfitting on this 2-D view, and which looks like it is underfitting? Explain the visual evidence.

*Your answer:* $C = 0.01$ looks like it is underfitting because the boundary is too smooth and fails to capture the class separation. $C = 100$ shows signs of overfitting as the boundary curves sharply around specific data points, whereas $C = 1$ appears to provide a well-balanced, smoother generalization boundary.

(d) Why is this 2-D plot only an APPROXIMATE illustration of the real (high-dimensional) SVM? What could it be misleading you about?

*Your answer:* This 2-D plot is only an approximation because PCA captures a total variance of only about $45.23\%$, meaning over half of the information from the original multi-dimensional feature space is lost. It can mislead us by making the data appear more linearly separable or simpler than it actually is in high dimensions.


---

## Part 4 — Systematic Hyperparameter Search

**Task 4.1 — Objective:** Run `GridSearchCV` over a C x gamma grid for an RBF-kernel SVM and select the best configuration using cross-validated F1-score.

**What to do:**
1. Define a parameter grid with at least 3 values of C and at least 3 values of gamma.
2. Run `GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='f1')` on your scaled **training** data.
3. Extract `gs.cv_results_` and reshape the mean test scores into a C x gamma grid.
4. Visualize as a heatmap (`imshow` or a pivoted DataFrame) with C and gamma labeled.
5. Refit the best estimator and report its performance on the held-out **test** set.

| Syntax | Description |
|---|---|
| `GridSearchCV(estimator, param_grid, cv, scoring)` | tries every combination via k-fold CV |
| `gs.best_params_` | best hyperparameter combination |
| `gs.cv_results_['mean_test_score']` | mean CV score per combination |


In [ ]:
# YOUR CODE HERE — define param_grid and run GridSearchCV

from sklearn.model_selection import GridSearchCV

# YOUR CODE HERE — define param_grid and run GridSearchCV[cite: 5, 9]
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1, 1]
}

gs = GridSearchCV(SVC(kernel='rbf', random_state=42), param_grid, cv=5, scoring='f1')
gs.fit(X_train_s, y_train)

print("Best Parameters:", gs.best_params_)
print("Best CV F1-Score:", gs.best_score_)

In [ ]:
# YOUR CODE HERE — heatmap of mean CV F1-score over the C x gamma grid

# YOUR CODE HERE — heatmap of mean CV F1-score over the C x gamma grid[cite: 5, 9]
scores = gs.cv_results_['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(8, 6))
sns.heatmap(scores, annot=True, fmt=".4f", cmap='viridis',
            xticklabels=param_grid['gamma'], yticklabels=param_grid['C'])
plt.xlabel("Gamma")
plt.ylabel("C")
plt.title("GridSearchCV Mean F1-Score Heatmap")
plt.show()

In [ ]:
# YOUR CODE HERE — refit best estimator, evaluate on test set

# YOUR CODE HERE — refit best estimator, evaluate on test set[cite: 5, 9]
best_svm_model = gs.best_estimator_
preds_grid = best_svm_model.predict(X_test_s)

print("Test Accuracy:", accuracy_score(y_test, preds_grid))
print("Test F1-Score:", f1_score(y_test, preds_grid))

**Questions to answer:**
(a) What were the best C and gamma values found, and the corresponding cross-validated F1-score?

*Your answer:* The best hyperparameters found were $C = 10$ and $\text{gamma} = 0.01$, with a corresponding best cross-validated F1-score of $0.5853$.

(b) Is there a clear "best region" in the heatmap, or do several combinations perform similarly? What does that tell you?

*Your answer:* Yes, the heatmap shows a distinct bright region around $C = 1$ and $C = 10$ with lower gamma values (scale and $0.01$) where scores peak, while higher gamma values (like $1$) perform poorly. This indicates that the model is sensitive to overly large gamma values which likely lead to overfitting.

(c) Did the cross-validated score and the final test-set score agree closely, or was there a gap? What would a large gap suggest?

*Your answer:* The cross-validated F1-score ($0.5853$) and the final test-set F1-score ($0.5565$) agreed quite closely with a minimal gap, indicating good generalization. A large gap would suggest overfitting to the training folds during cross-validation.

(d) Compare the tuned RBF model's test performance to the un-tuned RBF model from Part 2. Was tuning worth it?

*Your answer:*  The untuned RBF model from Part 2 had a test F1-score of $0.5562$, while the tuned RBF model achieved $0.5565$. Although the improvement is very slight, tuning helped identify stable hyperparameters ($C=10, \text{gamma}=0.01$) that optimize cross-validation performance.


---

## Part 5 — Class Imbalance and the Business Trade-off

**Task 5.1 — Objective:** Compare your best RBF-kernel SVM with and without `class_weight='balanced'`, and reason about which is more useful for the business.

**What to do:**
1. Using your best (C, gamma) from Part 4, train two RBF SVMs: `class_weight=None` and `class_weight='balanced'`.
2. Report Precision, Recall, F1-score and confusion matrix for both on the test set.
3. Plot an ROC curve or Precision-Recall curve for the balanced model.


In [ ]:
# YOUR CODE HERE — class_weight=None vs 'balanced' comparison

# YOUR CODE HERE — class_weight=None vs 'balanced' comparison[cite: 6, 9]
clf_def = SVC(kernel='rbf', C=gs.best_params_['C'], gamma=gs.best_params_['gamma'], probability=True, random_state=42)
clf_def.fit(X_train_s, y_train)
p_def = clf_def.predict(X_test_s)

clf_bal = SVC(kernel='rbf', C=gs.best_params_['C'], gamma=gs.best_params_['gamma'], class_weight='balanced', probability=True, random_state=42)
clf_bal.fit(X_train_s, y_train)
p_bal = clf_bal.predict(X_test_s)

print("Default -> Precision:", precision_score(y_test, p_def), "Recall:", recall_score(y_test, p_def))
print("Balanced -> Precision:", precision_score(y_test, p_bal), "Recall:", recall_score(y_test, p_bal))

In [ ]:
# YOUR CODE HERE — ROC or Precision-Recall curve

from sklearn.metrics import roc_curve, roc_auc_score

# YOUR CODE HERE — ROC or Precision-Recall curve[cite: 6, 9]
y_prob = clf_bal.predict_proba(X_test_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Balanced SVM)")
plt.legend()
plt.show()

**Questions to answer:**
(a) Precisely how did precision and recall move when you switched to `class_weight='balanced'`? Give the numbers.

*Your answer:* When switching to class_weight='balanced', precision dropped from $0.6275$ to $0.5070$, while recall significantly increased from $0.5000$ to $0.7781$, trading some precision to successfully capture a much higher proportion of actual churners.

(b) Is it more costly to miss an actual churner (false negative) or wrongly flag a loyal customer (false positive)? Which model would you deploy?

*Your answer:* In a telecom churn context, missing an actual churner (false negative) is typically more costly because the company loses a customer entirely without a chance to intervene. Therefore, I would deploy the balanced model to maximize recall and catch more potential churners.

(c) SVM does not directly output a "probability of churn" like Logistic Regression. What did you have to enable in `SVC()` to get probability-like scores?

*Your answer:* We had to enable probability=True in SVC() to generate probability estimates via Platt scaling.


---

## Part 6 — Model Showdown: SVM vs. Everything Else

**Task 6.1 — Objective:** Build a single comparison table across all five algorithms (Decision Tree, Random Forest, Logistic Regression, KNN, SVM) and select ONE final model for production.

**What to do:**
1. Build a table: one row per model, columns Accuracy / Precision / Recall / F1-score. Re-run earlier models here on the same split if you don't have saved numbers.
2. Produce ONE bar chart comparing F1-score across all five models.


In [ ]:
# YOUR CODE HERE — cross-model comparison table

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# YOUR CODE HERE — cross-model comparison table[cite: 6, 7, 9]
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Tuned SVM (Balanced)': clf_bal
}

comp = []
for name, m in models.items():
    if name != 'Tuned SVM (Balanced)':
        m.fit(X_train_s, y_train)
    preds = m.predict(X_test_s)
    comp.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds)
    })

df_comp = pd.DataFrame(comp).set_index('Model')
df_comp

In [ ]:
# YOUR CODE HERE — bar chart of F1-score across models

# YOUR CODE HERE — bar chart of F1-score across models[cite: 7, 9]
plt.figure(figsize=(9, 4))
df_comp['F1-Score'].plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("F1-Score Comparison Across Models")
plt.ylabel("F1-Score")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.show()

**Questions to answer:**
(a) Which model wins on F1-score? Which wins on recall? Are they the same model?

*Your answer:* The Tuned SVM (Balanced) wins on F1-score ($0.6139$) and Tuned SVM (Balanced) also wins on recall ($0.7781$). No, they are the same model in this case, as the balanced SVM achieves the highest score for both metrics.

(b) SVM is generally slower to train than Decision Tree/Random Forest here. Was the extra cost worth it?

*Your answer:* Yes, the extra computational cost was worth it because the Tuned Balanced SVM achieved a significantly higher recall ($0.7781$) compared to models like Random Forest ($0.4920$) and Decision Tree ($0.4920$), which is crucial for identifying churners in a business setting.

(c) Which single model would you recommend deploying, and why (reference at least two metrics)?

*Your answer:* I would recommend deploying the Tuned SVM (Balanced) because it achieves the highest F1-score ($0.6139$) and a superior recall ($0.7781$), ensuring that the telecom company successfully catches the vast majority of actual churners.


---

## Part 7 — Final Predictions and Wrap-Up

**Task 7.1 — Objective:** Generate a predictions file and write a short executive summary.

**What to do:**
1. Using your final chosen model, generate predictions on the test set and save `svm_predictions.csv` with true label, predicted label, and predicted churn probability/score.
2. Write a 5-8 sentence Executive Summary below: which model you chose, key metrics, the main trade-off you made in Part 5, and one limitation of your analysis.


In [ ]:
# YOUR CODE HERE — generate and save svm_predictions.csv

# YOUR CODE HERE — generate and save svm_predictions.csv[cite: 7, 9]
final_preds = clf_bal.predict(X_test_s)
final_probs = clf_bal.predict_proba(X_test_s)[:, 1]

df_res = pd.DataFrame({
    'True_Label': y_test.values,
    'Predicted_Label': final_preds,
    'Churn_Probability': final_probs
})

df_res.to_csv('svm_predictions.csv', index=False)
print("svm_predictions.csv saved successfully!")
df_res.head()

**Executive Summary**

*Your summary here.* For this project, I chose the Tuned Balanced SVM model as the final production model to predict customer churn. The model achieved a strong F1-score of $0.6139$ and an exceptional recall of $0.7781$, making it highly effective at catching actual churners. In Part 5, we deliberately made a key trade-off by switching to class_weight='balanced', which sacrificed some precision ($0.5070$) to significantly boost recall ($0.7781$), ensuring fewer churners are missed by the business. A primary limitation of this analysis is that it relies strictly on historical customer data, which may not fully capture future shifts in consumer behavior or market dynamics.  

**Final Reflection**
(a) What was the single most surprising result in this lab?

*Your answer:* The most surprising result was how severely the unscaled linear SVM suffered in convergence time and performance compared to the scaled version, emphasizing how critical geometric distance scaling is for SVMs

(b) If you had one more hour, what would you try next?

*Your answer:* If I had one more hour, I would implement SMOTE (Synthetic Minority Over-sampling Technique) to handle class imbalance instead of relying solely on class weights, and explore advanced hyperparameter tuning across a broader range of kernels.
